# Optimization Note 6: Robustness — Inertia Correction, Regularization, and Restoration

**Goal:** Understand the machinery that makes the difference between a textbook IPM and a production solver.

## Why Robustness Matters

The IPM from Notes 4-5 works on well-conditioned problems with good starting points. Production solvers face:

1. **Indefinite Hessians** at non-convex points → Newton step is wrong direction
2. **Singular KKT matrices** from degenerate constraints → factorization fails
3. **Deeply infeasible starting points** → large steps can make things worse
4. **Near-zero pivots** in the linear solver → inaccurate search direction

This note covers the three key robustness mechanisms: inertia correction, regularization, and restoration.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=6, suppress=True)

## 1. Inertia Correction

### The Problem

The KKT matrix should have **inertia $(n, m, 0)$**: $n$ positive eigenvalues (from $H + \Sigma$), $m$ negative eigenvalues (from the constraint block), and zero null eigenvalues.

If the Hessian $H$ has negative eigenvalues (non-convex region), the $(1,1)$ block $H + \Sigma$ may not be positive definite, giving wrong inertia.

### The Fix: Primal Regularization

Add $\delta_w I$ to the $(1,1)$ block:

$$\begin{bmatrix} H + \Sigma + \delta_w I & J^T \\ J & -\delta_c I \end{bmatrix}$$

- **$\delta_w > 0$**: shifts eigenvalues of $(1,1)$ block positive
- **$\delta_c > 0$**: ensures the $(2,2)$ block is strictly negative (handles rank-deficient $J$)

### ripopt's Strategy

1. Try $\delta_w = 0$: factor and check inertia (this is free from LDL^T — see Linear Note 3)
2. If inertia is wrong, start with $\delta_w = 10^{-4}$
3. If still wrong, multiply by 4: $\delta_w \leftarrow 4 \delta_w$
4. Repeat up to 15 times
5. Warm-start: use $\delta_w / 4$ from the previous iteration's successful value

In [2]:
def compute_inertia(M, tol=1e-10):
    """Compute inertia (n+, n-, n0) of a symmetric matrix."""
    eigvals = np.linalg.eigvalsh(M)
    n_pos = np.sum(eigvals > tol)
    n_neg = np.sum(eigvals < -tol)
    n_zero = len(eigvals) - n_pos - n_neg
    return (n_pos, n_neg, n_zero)


def inertia_correction(H, Sigma, J, n, m, delta_c=1e-8, max_attempts=15):
    """Find delta_w such that KKT has inertia (n, m, 0).
    
    Returns: corrected KKT matrix, delta_w used.
    """
    target = (n, m, 0)
    
    def build_kkt(delta_w):
        K = np.zeros((n + m, n + m))
        K[:n, :n] = H + np.diag(Sigma) + delta_w * np.eye(n)
        K[:n, n:] = J.T
        K[n:, :n] = J
        K[n:, n:] = -delta_c * np.eye(m)
        return K
    
    # Try without perturbation
    K = build_kkt(0.0)
    inertia = compute_inertia(K)
    if inertia == target:
        return K, 0.0, [(0.0, inertia)]
    
    # Iterative correction
    delta_w = 1e-4
    growth = 4.0
    trials = [(0.0, inertia)]
    
    for attempt in range(max_attempts):
        K = build_kkt(delta_w)
        inertia = compute_inertia(K)
        trials.append((delta_w, inertia))
        
        if inertia == target:
            return K, delta_w, trials
        
        delta_w *= growth
    
    return K, delta_w, trials  # return best effort

In [3]:
# Example: non-convex Hessian
n, m = 3, 1

# Hessian with a negative eigenvalue (non-convex region)
H = np.array([
    [ 2.0,  0.0,  0.0],
    [ 0.0, -1.0,  0.0],  # negative eigenvalue!
    [ 0.0,  0.0,  3.0]
])
Sigma = np.array([0.1, 0.1, 0.1])  # barrier diagonal (small)
J = np.array([[1.0, 1.0, 1.0]])     # constraint Jacobian

K_corrected, delta_w, trials = inertia_correction(H, Sigma, J, n, m)

print(f"Target inertia: ({n}, {m}, 0)")
print()
for dw, inertia in trials:
    status = "✓" if inertia == (n, m, 0) else "✗"
    print(f"  delta_w = {dw:.1e}  inertia = {inertia}  {status}")

print(f"\nCorrection needed: delta_w = {delta_w:.1e}")
print(f"This shifts H's negative eigenvalue ({-1.0}) to {-1.0 + delta_w:.4f}")

Target inertia: (3, 1, 0)

  delta_w = 0.0e+00  inertia = (np.int64(3), np.int64(1), np.int64(0))  ✓

Correction needed: delta_w = 0.0e+00
This shifts H's negative eigenvalue (-1.0) to -1.0000


## 2. Why Inertia Comes Free

This is where the linear algebra tutorial connects directly to optimization.

In Linear Note 3, we saw that LDL^T factorization gives the matrix's inertia for free: just count the signs of the diagonal entries of $D$. So checking inertia after factorization costs **zero additional work**.

If inertia is wrong, we add $\delta_w I$ and **refactor** — the symbolic analysis is reused, only the numeric factorization is redone. In ripopt, re-factorization after inertia correction typically costs less than the original factorization because the symbolic structure is cached.

## 3. The Restoration Phase

### The Problem

Sometimes the line search fails completely: no step length $\alpha$ produces acceptable decrease. This happens when:
- The current point is deeply infeasible
- The linearization badly misrepresents the constraint surface
- The search direction points away from feasibility

### The Fix: Temporarily Forget the Objective

Enter **restoration mode**: minimize constraint violation without caring about the objective:

$$\min_x \; \|c(x)\|^2$$

Once we've reduced the constraint violation enough, return to the normal IPM.

### ripopt's Two-Phase Restoration

**Phase 1: Gauss-Newton** (fast, works 90%+ of the time)
- Solve $J^T J \, \Delta x = -J^T c(x)$ (least-squares on constraint violation)
- With Levenberg-Marquardt regularization: $(J^T J + \epsilon I) \, \Delta x = -J^T c(x)$
- Line search on $\|c(x)\|^2$

**Phase 2: NLP Restoration** (robust fallback)
- Formulate a new NLP with penalty on infeasibility
- Solve with the same IPM (but with restoration disabled to prevent infinite recursion)

In [4]:
def gauss_newton_restoration(c, J, x0, tol=1e-6, max_iter=50):
    """Phase 1 restoration: minimize ||c(x)||^2 via Gauss-Newton."""
    x = np.array(x0, dtype=float)
    history = []
    
    for k in range(max_iter):
        cv = c(x)
        violation = np.linalg.norm(cv)
        history.append({'x': x.copy(), 'violation': violation})
        
        if violation < tol:
            break
        
        Jx = J(x)
        
        # Gauss-Newton: (J^T J + eps*I) dx = -J^T c
        JtJ = Jx.T @ Jx
        eps = 1e-8
        for _ in range(10):
            try:
                dx = np.linalg.solve(JtJ + eps * np.eye(len(x)), -Jx.T @ cv)
                break
            except np.linalg.LinAlgError:
                eps *= 10
        
        # Line search on ||c||^2
        alpha = 1.0
        for _ in range(20):
            x_trial = x + alpha * dx
            if np.linalg.norm(c(x_trial)) < violation:
                break
            alpha *= 0.5
        
        x = x + alpha * dx
    
    return x, history

In [5]:
# Demonstrate restoration on a deeply infeasible starting point
# Constraint: x^2 + y^2 = 1 (unit circle)

c_circle = lambda x: np.array([x[0]**2 + x[1]**2 - 1.0])
J_circle = lambda x: np.array([[2*x[0], 2*x[1]]])

# Start far from the circle
x0 = np.array([5.0, 5.0])
print(f"Starting violation: ||c(x0)|| = {np.linalg.norm(c_circle(x0)):.4f}")

x_restored, hist = gauss_newton_restoration(c_circle, J_circle, x0)

print(f"Restored to: x = {x_restored}")
print(f"Final violation: {np.linalg.norm(c_circle(x_restored)):.2e}")
print(f"\nRestoration convergence:")
for i, h in enumerate(hist):
    print(f"  iter {i}: ||c|| = {h['violation']:.6e}")

Starting violation: ||c(x0)|| = 49.0000
Restored to: x = [0.707107 0.707107]
Final violation: 4.87e-08

Restoration convergence:
  iter 0: ||c|| = 4.900000e+01
  iter 1: ||c|| = 1.200500e+01
  iter 2: ||c|| = 2.770473e+00
  iter 3: ||c|| = 5.089230e-01
  iter 4: ||c|| = 4.291184e-02
  iter 5: ||c|| = 4.414147e-04
  iter 6: ||c|| = 4.869133e-08


## 4. Second-Order Correction (SOC)

A subtler issue: the linearized constraints $c(x_k) + J(x_k) d$ may poorly approximate the actual constraints $c(x_k + d)$ for curved constraint surfaces.

The **second-order correction** fixes this by solving a corrected KKT system:

1. Compute trial point: $x^+ = x_k + \alpha d_k$
2. If constraint violation increased: solve for correction $\hat{d}$ using the actual constraint violation at $x^+$
3. Try $x^+ + \hat{d}$ instead

This reuses the existing factorization (just a new back-solve) so it's cheap. ripopt applies SOC during every backtracking step where constraint violation increases.

In [6]:
# Demonstrate SOC on a curved constraint
# min x1 + x2  s.t. x1^2 + x2^2 = 2

def illustrate_soc():
    x = np.array([1.0, 1.0])  # on the circle x^2+y^2=2
    
    # Objective: min x1 + x2
    grad = np.array([1.0, 1.0])
    H = np.zeros((2, 2))  # linear objective
    c_val = np.array([x[0]**2 + x[1]**2 - 2.0])
    J = np.array([[2*x[0], 2*x[1]]])
    
    # Solve KKT for Newton direction
    KKT = np.zeros((3, 3))
    KKT[:2, :2] = H + 0.01 * np.eye(2)  # slight regularization
    KKT[:2, 2:] = J.T
    KKT[2:, :2] = J
    rhs = np.concatenate([-grad, -c_val])
    sol = np.linalg.solve(KKT, rhs)
    dx = sol[:2]
    
    # Take step
    x_trial = x + dx
    c_trial = x_trial[0]**2 + x_trial[1]**2 - 2.0
    
    # SOC: correct using actual violation at trial point
    rhs_soc = np.concatenate([np.zeros(2), -np.array([c_trial])])
    sol_soc = np.linalg.solve(KKT, rhs_soc)
    dx_soc = sol_soc[:2]
    x_corrected = x_trial + dx_soc
    c_corrected = x_corrected[0]**2 + x_corrected[1]**2 - 2.0
    
    print(f"Starting point: x = {x}, c = {c_val[0]:.6f}")
    print(f"After Newton:   x = {x_trial}, c = {c_trial:.6f}")
    print(f"After SOC:      x = {x_corrected}, c = {c_corrected:.6f}")
    print(f"\nSOC reduced constraint violation by {abs(c_trial)/max(abs(c_corrected),1e-15):.0f}x")

illustrate_soc()

Starting point: x = [1. 1.], c = 0.000000
After Newton:   x = [1. 1.], c = 0.000000
After SOC:      x = [1. 1.], c = 0.000000

SOC reduced constraint violation by 0x


## 5. Putting the Robustness Pieces Together

Here's how these mechanisms interact in a single iteration of ripopt:

```
1. Assemble KKT matrix (H + Σ + δ_w·I, J, -δ_c·I)
2. Factor with LDL^T (multifrontal for large, dense for small)
3. Check inertia from D diagonal:
   - If (n, m, 0): proceed ✓
   - If wrong: increase δ_w, go to step 1
4. Solve for (Δx, Δy)
5. Fraction-to-boundary → α_max
6. Filter line search:
   a. Try α = α_max
   b. Check filter acceptance
   c. If rejected and θ increased: try SOC
   d. If SOC works: accept
   e. If not: backtrack α ← α/2, repeat
7. If all α exhausted (40 backtracks):
   → Enter RESTORATION
   a. Phase 1: Gauss-Newton on ||c(x)||²
   b. If GN fails twice: Phase 2: NLP restoration
   c. Return to step 1 with restored x
8. Accept step, update x, y, z
9. Update μ (free or fixed mode)
```

In [7]:
# Demonstrate the full flow on a tricky problem
# Minimize x^3 subject to x >= 1  (non-convex objective, active bound)
#
# At x=1: f'=3, f''=6 (convex here, but x^3 is non-convex for x<0)
# Starting at x=2, the solver should find x=1

# Use our barrier solver from Note 3 to show the approach
x = np.array([2.0])
mu = 0.1

print(f"{'Iter':>4} {'x':>10} {'f(x)':>10} {'mu':>10} {'z':>10} {'compl':>10}")
print("-" * 58)

for k in range(20):
    s = x[0] - 1.0  # slack to lower bound
    z = mu / s       # bound multiplier estimate
    
    # Barrier gradient: f'(x) - mu/(x-1)
    g = 3 * x[0]**2 - mu / s
    # Barrier Hessian: f''(x) + mu/(x-1)^2
    H = 6 * x[0] + mu / s**2
    
    compl = z * s
    print(f"{k:>4} {x[0]:>10.6f} {x[0]**3:>10.6f} {mu:>10.2e} {z:>10.4f} {compl:>10.2e}")
    
    if abs(g) < 1e-10 and compl < 1e-10:
        break
    
    # Newton step on barrier
    dx = -g / H
    
    # Fraction to boundary
    alpha = 1.0
    if dx < 0:
        alpha = min(alpha, -0.99 * s / dx)
    
    x[0] += alpha * dx
    
    # Update mu
    s_new = x[0] - 1.0
    z_new = mu / s_new
    mu = min(0.2 * mu, (z_new * s_new) / 10.0)
    mu = max(mu, 1e-12)

print(f"\nSolution: x = {x[0]:.8f} (expected: 1.0)")

Iter          x       f(x)         mu          z      compl
----------------------------------------------------------
   0   2.000000   8.000000   1.00e-01     0.1000   1.00e-01
   1   1.016529   1.050411   1.00e-02     0.6050   1.00e-02
   2   1.000165   1.000496   1.00e-03     6.0500   1.00e-03
   3   1.000249   1.000746   1.00e-04     0.4023   1.00e-04
   4   1.000002   1.000007   1.00e-05     4.0229   1.00e-05
   5   1.000003   1.000009   1.00e-06     0.3207   1.00e-06
   6   1.000000   1.000000   1.00e-07     3.2074   1.00e-07
   7   1.000000   1.000000   1.00e-08     0.3013   1.00e-08
   8   1.000000   1.000000   1.00e-09     3.0126   1.00e-09
   9   1.000000   1.000000   1.00e-10     0.3000   1.00e-10
  10   1.000000   1.000000   1.00e-11     3.0000   1.00e-11
  11   1.000000   1.000000   1.00e-12     0.3000   1.00e-12
  12   1.000000   1.000000   1.00e-12    30.0240   1.00e-12
  13   1.000000   1.000000   1.00e-12    15.8021   1.00e-12
  14   1.000000   1.000000   1.00e-12    

## What We've Learned

1. **Inertia correction** adds $\delta_w I$ to the $(1,1)$ block until the KKT matrix has the right eigenvalue signature — detected for free from LDL^T
2. **Restoration** temporarily abandons the objective to reduce constraint violation when the line search fails
3. **Second-order correction** cheaply improves the step along curved constraints using one extra back-solve
4. These mechanisms interact: inertia → factor → search → SOC → restoration, with fallbacks at each stage
5. The warm-starting of $\delta_w$ across iterations avoids unnecessary re-correction

## What's Next

In Note 7, we trace a complete ripopt solve step by step, showing how all the pieces from this series come together in the actual implementation.

---

*This is Optimization Note 6 in a series building from Newton's method to the interior point optimizer [ripopt](https://github.com/jkitchin/ripopt).*